# Change Data Capture

In [5]:
!pip cache purge --quiet

In [6]:
!pip install Faker==37.12.0 --quiet

In [7]:
import time
import random

from datetime import datetime, timedelta
from faker import Faker
from faker.providers import BaseProvider
from pymongo import MongoClient

In [8]:
# Setup reproducibility
SEED = 42
REFERENCE_DATE = datetime(2024, 10, 1, 12, 0, 0)

fake = Faker()
Faker.seed(SEED)
random.seed(SEED)

In [9]:
# Custom provider for CRM stages

class CustomStageProvider(BaseProvider):
    STAGES = [
        "new lead",
        "contacted",
        "qualified",
        "proposal sent",
        "negotiation",
        "won",
        "lost"
    ]
    def stage(self):
        return self.random_element(self.STAGES)

fake.add_provider(CustomStageProvider)

In [13]:
# MongoDB Connection
try:
    client = MongoClient("mongodb+srv://admin:<password>@<host>/?appName=Cluster0")
    db = client["crm_db"]
    print("Connected to MongoDB successfully.")
except Exception as e:
    print(f"Could not connect: {e}")
    exit(1)

# Clean collections
print("Cleaning existing data...")
for coll in ["customers", "orders", "products"]:
    count = db[coll].delete_many({}).deleted_count
    print(f"  Deleted {count} documents from {coll}")

Connected to MongoDB successfully.
Cleaning existing data...
  Deleted 0 documents from customers
  Deleted 0 documents from orders
  Deleted 0 documents from products


In [14]:
# Parameters
NUM_CUSTOMERS = 50
NUM_PRODUCTS = 20
NUM_ORDERS = 100

In [22]:
print("Phase 1: Initial Data Load")

# Generate Customers (store in list to maintain order)
customers = []
for i in range(NUM_CUSTOMERS):
    customers.append({
        "customer_id": i,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.email(),
        "company": fake.company(),
        "stage": fake.stage(),
        "created_at": fake.date_time_this_year()
    })
db.customers.insert_many(customers)
print(f"Inserted {NUM_CUSTOMERS} customers")

# Generate Products (store in list to maintain order)
products = []
for i in range(NUM_PRODUCTS):
    products.append({
        "product_id": i,
        "name": fake.bs().title(),
        "price": round(random.uniform(10.0, 1000.0), 2),
        "category": random.choice(["software", "hardware", "service"]),
        "created_at": fake.date_time_this_year()
    })
db.products.insert_many(products)
print(f"Inserted {NUM_PRODUCTS} products")

# Generate Orders
orders = []
for i in range(NUM_ORDERS):
    cust = customers[i % NUM_CUSTOMERS]
    prod = products[i % NUM_PRODUCTS]
    qty = random.randint(1, 10)
    orders.append({
        "order_id": i,
        "customer_email": cust["email"],
        "product_name": prod["name"],
        "quantity": qty,
        "total_price": round(prod["price"] * qty, 2),
        "order_date": fake.date_time_this_year()
    })
db.orders.insert_many(orders)
print(f"Inserted {NUM_ORDERS} orders")

print("Phase 1 Complete: Initial load finished")
print("\nPAUSE HERE: Set up SingleStore pipeline, then press Enter...")
input()

print("Waiting 10 seconds for SingleStore initial sync...")
time.sleep(10)

Phase 1: Initial Data Load
Inserted 50 customers
Inserted 20 products
Inserted 100 orders
Phase 1 Complete: Initial load finished

PAUSE HERE: Set up SingleStore pipeline, then press Enter...


Waiting 10 seconds for SingleStore initial sync...


In [32]:
print("Phase 2: Simulating CDC Changes")

# Reset seed for deterministic CDC changes
Faker.seed(SEED + 1)
random.seed(SEED + 1)

# Use fixed timestamp offset for deterministic timestamps
CDC_TIMESTAMP = REFERENCE_DATE + timedelta(days = 100)

# New customers (deterministic)
new_customers = []
for i in range(5):
    new_customers.append({
        "customer_id": NUM_CUSTOMERS + i,
        "first_name": fake.first_name(),
        "last_name": fake.last_name(),
        "email": fake.email(),
        "company": fake.company(),
        "stage": "new lead",
        "created_at": CDC_TIMESTAMP + timedelta(minutes = i)
    })
result = db.customers.insert_many(new_customers)
print(f"Inserted {len(result.inserted_ids)} new customers")

# Progress customer stages (deterministic selection)
# Sort by customer_id for deterministic order
customers_to_update = list(db.customers.find(
    {"stage": {"$ne": "won"}}
).sort("customer_id", 1).limit(3))

updated_count = 0
for cust in customers_to_update:
    db.customers.update_one(
        {"_id": cust["_id"]},
        {"$set": {
            "stage": "won",
            "updated_at": CDC_TIMESTAMP + timedelta(minutes = 10 + updated_count)
        }}
    )
    updated_count += 1
print(f"Updated {updated_count} customers")

# New orders (deterministic)
# Get customers and products in deterministic order
all_customers = list(db.customers.find().sort("customer_id", 1))
all_products = list(db.products.find().sort("product_id", 1))

new_orders = []
for i in range(10):
    # Use modulo for deterministic cycling through lists
    cust = all_customers[i % len(all_customers)]
    prod = all_products[i % len(all_products)]
    qty = random.randint(1, 5)
    new_orders.append({
        "order_id": NUM_ORDERS + i,
        "customer_email": cust["email"],
        "product_name": prod["name"],
        "quantity": qty,
        "total_price": round(prod["price"] * qty, 2),
        "order_date": CDC_TIMESTAMP + timedelta(minutes = 20 + i)
    })
result = db.orders.insert_many(new_orders)
print(f"Inserted {len(result.inserted_ids)} new orders")

# Change product prices (deterministic)
products_to_update = list(db.products.find().sort("product_id", 1).limit(3))

price_update_count = 0
for prod in products_to_update:
    new_price = round(prod["price"] * random.uniform(0.95, 1.05), 2)
    db.products.update_one(
        {"_id": prod["_id"]},
        {"$set": {
            "price": new_price,
            "updated_at": CDC_TIMESTAMP + timedelta(minutes = 30 + price_update_count)
        }}
    )
    price_update_count += 1
print(f"Updated {price_update_count} product prices")

# Remove products (deterministic)
products_to_delete = list(db.products.find().sort("product_id", 1).limit(2))

deleted_count = 0
for prod in products_to_delete:
    db.products.delete_one({"_id": prod["_id"]})
    deleted_count += 1
print(f"Deleted {deleted_count} products")

print("Phase 2 Complete: CDC changes applied")

Phase 2: Simulating CDC Changes
Inserted 5 new customers
Updated 3 customers
Inserted 10 new orders
Updated 3 product prices
Deleted 2 products
Phase 2 Complete: CDC changes applied


In [34]:
# Print expected results
print("Expected Results:")
print(f"  Customers: 50 -> {db.customers.count_documents({})} (should be 55)")
print(f"  Products:  20 -> {db.products.count_documents({})} (should be 18)")
print(f"  Orders:    100 -> {db.orders.count_documents({})} (should be 110)")

print("Waiting 15 seconds for CDC propagation...")
time.sleep(15)

Expected Results:
  Customers: 50 -> 55 (should be 55)
  Products:  20 -> 18 (should be 18)
  Orders:    100 -> 110 (should be 110)
Waiting 15 seconds for CDC propagation...
